## AEF's uncertainty thresholds 

In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import ShuffleSplit
from src.utils import entropy_func, uncertainty_error_overlap

The objective of this code is to create a file with all optmized uncertainty thresholds used in AEFS's methodology. In this way, we will be able to create only all the automatically extracted from these uncertainty threshold and use them later to create our graph.

In [ ]:
task ='brats'
data = np.load(f'./data/{task}.npz')

y = data['y']
p_hat = data['p_hat']
y_hat = data['y_hat'] # This line should be COMMENTED in case y_hat is not given

n = 50 # Number of ShuffleSplits
min_prop = 2/len(y) # Minimum prportion
tuning_size_arrays = np.logspace(np.log10(min_prop), np.log10(0.8), num=25) # Tuning sizes proportion
u_threshs = [] # Uncertainty thresholds to be used in AEF

for tuning_size in tuning_size_arrays:
    for i in range(n):
        shuffle_split = ShuffleSplit(n_splits=1, test_size=(1-tuning_size), random_state=i)
        for train_index, test_index in shuffle_split.split(y):
            p_hat_train = [p_hat[i] for i in train_index]
            y_hat_train = [y_hat[i] for i in train_index] # This line should be COMMENTED in case y_hat is not given
            y_train = [y[i] for i in train_index]

            uncertainty_masks = entropy_func(p_hat_train)
            #u_thresh = uncertainty_error_overlap(y_train,p_hat_train,uncertainty_masks, T=0.5) # In case y_hat is not given. Change T (threshold) for MSWML (0.35)
            u_thresh = uncertainty_error_overlap(y_train,y_hat_train,uncertainty_masks, T=0.5) # This line is for BRaTS (y_hat given), if y_hat is not provided, use the line above
            u_threshs.append(u_thresh)

In [ ]:
with open(f'./data/{task}_u_threshs.pkl', 'wb') as file:
    # Save the list to the file
    pickle.dump(u_threshs, file)